### In Class April 28 Census Data

In [ ]:
# pip install catboost

  Using cached graphviz-0.21-py3-none-any.whl.metadata (12 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 28.9/28.9 MB 102.4 MB/s  0:00:00 eta 0:00:01
Using cached graphviz-0.21-py3-none-any.whl (47 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [catboost]1/2 [catboost]
Note: you may need to restart the kernel to use updated packages.


In [ ]:
# import libraries
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split,StratifiedKFold,cross_validate
from sklearn.naive_bayes import GaussianNB
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler,OneHotEncoder,FunctionTransformer
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score,precision_score,recall_score,f1_score,balanced_accuracy_score
from xgboost import XGBClassifier
from catboost import CatBoostClassifier
import optuna

### Importing, inspecting, target encoding & splitting data

In [8]:
# load adult census data, inspect it, clean target, and split it
adult_data=pd.read_csv("adult.csv",na_values=["?"," ?"])

adult_data["income"]=adult_data["income"].astype(str).str.strip()
adult_data["income"]=adult_data["income"].map({
    "<=50K":0,
    ">50K":1,
})

print(adult_data.head())
print()
print(adult_data["income"].value_counts())
print()
print(adult_data.isna().sum())

X_adult=adult_data.drop(["income","fnlwgt","education"],axis=1)
y_adult=adult_data["income"]

X_train_adult,X_test_adult,y_train_adult,y_test_adult=train_test_split(
    X_adult,y_adult,test_size=0.20,stratify=y_adult,random_state=42
)

   age  workclass  fnlwgt     education  educational-num      marital-status  \
0   25    Private  226802          11th                7       Never-married   
1   38    Private   89814       HS-grad                9  Married-civ-spouse   
2   28  Local-gov  336951    Assoc-acdm               12  Married-civ-spouse   
3   44    Private  160323  Some-college               10  Married-civ-spouse   
4   18        NaN  103497  Some-college               10       Never-married   

          occupation relationship   race  gender  capital-gain  capital-loss  \
0  Machine-op-inspct    Own-child  Black    Male             0             0   
1    Farming-fishing      Husband  White    Male             0             0   
2    Protective-serv      Husband  White    Male             0             0   
3  Machine-op-inspct      Husband  Black    Male          7688             0   
4                NaN    Own-child  White  Female             0             0   

   hours-per-week native-country  inco

### Defining functions

In [5]:
# helper function for cross validation comparison
def compare_models_cv(models,X_train,y_train,cv):
    cv_results=[]
    for name,model in models.items():
        scores=cross_validate(model,X_train,y_train,cv=cv,scoring=["accuracy","precision","recall","f1","balanced_accuracy"])
        cv_results.append({
            "model":name,
            "cv_accuracy":scores["test_accuracy"].mean(),
            "cv_precision":scores["test_precision"].mean(),
            "cv_recall":scores["test_recall"].mean(),
            "cv_f1":scores["test_f1"].mean(),
            "cv_balanced_accuracy":scores["test_balanced_accuracy"].mean()
        })
    return pd.DataFrame(cv_results).sort_values("cv_balanced_accuracy",ascending=False)

In [6]:
# helper function for threshold tuning
def evaluate_thresholds(y_true,probs,thresholds=None):
    if thresholds is None:
        thresholds=np.linspace(0.1,0.9,17) # change 17 to higher number to test more thresholds
    rows=[]
    for t in thresholds:
        preds=(probs[:,1]>=t).astype(int)
        rows.append({
            "threshold":t,
            "precision":precision_score(y_true,preds,zero_division=0),
            "recall":recall_score(y_true,preds,zero_division=0),
            "f1":f1_score(y_true,preds,zero_division=0),
            "balanced_accuracy":balanced_accuracy_score(y_true,preds)
        })
    return pd.DataFrame(rows)

In [7]:
# helper function for held out test evaluation
def evaluate_models_test(models,X_train,X_test,y_train,y_test):
    trained_models={}
    test_results=[]
    for name,model in models.items():
        model.fit(X_train,y_train)
        trained_models[name]=model
        preds=model.predict(X_test)
        test_results.append({
            "model":name,
            "test_accuracy":accuracy_score(y_test,preds),
            "test_precision":precision_score(y_test,preds,zero_division=0),
            "test_recall":recall_score(y_test,preds,zero_division=0),
            "test_f1":f1_score(y_test,preds,zero_division=0),
            "test_balanced_accuracy":balanced_accuracy_score(y_test,preds)
        })
    return trained_models,pd.DataFrame(test_results).sort_values("test_balanced_accuracy",ascending=False)

In [22]:
# helper for converting sparse matrices to dense arrays
def to_dense(x):
    return x.toarray() if hasattr(x, "toarray") else x

### Identifying categorical/numerical & defining CV

In [10]:
# identify numeric and categorical columns and define cross validation
numeric_features_adult=X_train_adult.select_dtypes(include=["int64","float64"]).columns.tolist()
categorical_features_adult=X_train_adult.select_dtypes(include=["object"]).columns.tolist()

print("numeric columns:",numeric_features_adult)
print("categorical columns:",categorical_features_adult)

cv=StratifiedKFold(n_splits=5,shuffle=True,random_state=42)

numeric columns: ['age', 'educational-num', 'capital-gain', 'capital-loss', 'hours-per-week']
categorical columns: ['workclass', 'marital-status', 'occupation', 'relationship', 'race', 'gender', 'native-country']


### Transformers/pipelines & defining model

In [23]:
# define preprocessors and standardized adult models
numeric_transformer_scaled=Pipeline([
    ("imputer",SimpleImputer(strategy="median")),
    ("scaler",StandardScaler())
])

numeric_transformer_unscaled=Pipeline([
    ("imputer",SimpleImputer(strategy="median"))
])

categorical_transformer=Pipeline([
    ("imputer",SimpleImputer(strategy="most_frequent")),
    ("onehot",OneHotEncoder(handle_unknown="ignore"))
])

preprocessor_adult_scaled=ColumnTransformer([
    ("num",numeric_transformer_scaled,numeric_features_adult),
    ("cat",categorical_transformer,categorical_features_adult)
])

preprocessor_adult_unscaled=ColumnTransformer([
    ("num",numeric_transformer_unscaled,numeric_features_adult),
    ("cat",categorical_transformer,categorical_features_adult)
])

models_adult={
    "Naive Bayes":Pipeline([
    ("preprocessor",preprocessor_adult_unscaled),
    ("to_dense",FunctionTransformer(to_dense)),
    ("model",GaussianNB())
]),
    "Logistic Regression":Pipeline([
        ("preprocessor",preprocessor_adult_scaled),
        ("model",LogisticRegression(max_iter=2000))
    ]),
    "Random Forest":Pipeline([
        ("preprocessor",preprocessor_adult_unscaled),
        ("model",RandomForestClassifier(random_state=42))
    ]),
    "XGBoost":Pipeline([
        ("preprocessor",preprocessor_adult_unscaled),
        ("model",XGBClassifier(random_state=42,eval_metric="logloss"))
    ])
}

### Model comparison with cross validation

In [24]:
# compare adult models with standardized preprocessing
compare_models_cv(models_adult,X_train_adult,y_train_adult,cv)

,model,cv_accuracy,cv_precision,cv_recall,cv_f1,cv_balanced_accuracy
0,Naive Bayes,0.787884,0.537136,0.826292,0.650936,0.801048
3,XGBoost,0.872981,0.778711,0.655578,0.711813,0.798469
2,Random Forest,0.846313,0.706178,0.612900,0.656136,0.766314
1,Logistic Regression,0.850945,0.730510,0.598032,0.657591,0.764263


### Baseline model evaluation on test data

In [25]:
# fit standardized adult models and evaluate on held-out test set
trained_models_adult,test_results_adult=evaluate_models_test(
    models_adult,X_train_adult,X_test_adult,y_train_adult,y_test_adult
)
test_results_adult

,model,test_accuracy,test_precision,test_recall,test_f1,test_balanced_accuracy
0,Naive Bayes,0.788515,0.537841,0.826775,0.651719,0.801626
3,XGBoost,0.874092,0.779798,0.660393,0.715146,0.800860
2,Random Forest,0.852697,0.726448,0.616766,0.667129,0.771847
1,Logistic Regression,0.852185,0.737513,0.593670,0.657820,0.763596


### Threshold tuning baseline models

In [ ]:
# threshold tuning on standardized adult models (including default 0.5)

print("Logistic Regression")
probs_adult_lr = trained_models_adult["Logistic Regression"].predict_proba(X_test_adult)
df_lr = evaluate_thresholds(y_test_adult, probs_adult_lr)
top_lr = df_lr.sort_values("f1", ascending=False).head()
default_lr = df_lr[df_lr["threshold"] == 0.5]
print(pd.concat([top_lr, default_lr]))
print()

print("Naive Bayes")
probs_adult_nb = trained_models_adult["Naive Bayes"].predict_proba(X_test_adult)
df_nb = evaluate_thresholds(y_test_adult, probs_adult_nb)
top_nb = df_nb.sort_values("f1", ascending=False).head()
default_nb = df_nb[df_nb["threshold"] == 0.5]
print(pd.concat([top_nb, default_nb]))
print()

print("Random Forest")
probs_adult_rf = trained_models_adult["Random Forest"].predict_proba(X_test_adult)
df_rf = evaluate_thresholds(y_test_adult, probs_adult_rf)
top_rf = df_rf.sort_values("f1", ascending=False).head()
default_rf = df_rf[df_rf["threshold"] == 0.5]
print(pd.concat([top_rf, default_rf]))
print()

print("XGBoost")
probs_adult_xgb = trained_models_adult["XGBoost"].predict_proba(X_test_adult)
df_xgb = evaluate_thresholds(y_test_adult, probs_adult_xgb)
top_xgb = df_xgb.sort_values("f1", ascending=False).head()
default_xgb = df_xgb[df_xgb["threshold"] == 0.5]
print(pd.concat([top_xgb, default_xgb]))

Logistic Regression
   threshold  precision    recall        f1  balanced_accuracy
5       0.35   0.651498  0.734816  0.690653           0.805572
4       0.30   0.613735  0.783576  0.688334           0.814207
6       0.40   0.681973  0.686056  0.684009           0.792699
3       0.25   0.570173  0.832335  0.676752           0.817460
7       0.45   0.711566  0.639435  0.673575           0.778943
8       0.50   0.737513  0.593670  0.657820           0.763596

Naive Bayes
    threshold  precision    recall        f1  balanced_accuracy
14       0.80   0.589666  0.746792  0.658992           0.791644
13       0.75   0.575272  0.768178  0.657875           0.794868
15       0.85   0.609810  0.712575  0.657199           0.784561
16       0.90   0.638161  0.676647  0.656840           0.777968
12       0.70   0.565810  0.781437  0.656368           0.796384
8        0.50   0.537841  0.826775  0.651719           0.801626

Random Forest
   threshold  precision    recall        f1  balanced_accuracy


### Tuning baseline models with Optuna

#### show best tuning results

tuning_results = pd.DataFrame([
    {"model":"Naive Bayes","best_cv_f1":study_nb.best_value,**study_nb.best_params},
    {"model":"Logistic Regression","best_cv_f1":study_lr.best_value,**study_lr.best_params},
    {"model":"Random Forest","best_cv_f1":study_rf.best_value,**study_rf.best_params},
    {"model":"XGBoost","best_cv_f1":study_xgb.best_value,**study_xgb.best_params}
])

tuning_results

print("NB:", study_nb.best_params)
print("LR:", study_lr.best_params)
print("RF:", study_rf.best_params)
print("XGB:", study_xgb.best_params)

### Best parameters (from Optuna above)

NB: 'var_smoothing': 2.0413057857909265e-09
LR: 'C': 0.140759458974037, 'solver': 'lbfgs', 'class_weight': 'balanced'}
RF: 'n_estimators': 400, 'max_depth': 23, 'min_samples_split': 15, 'min_samples_leaf': 3, 'max_features': None, 'class_weight': 'balanced_subsample'
XGB: 'n_estimators': 800, 'max_depth': 5, 'learning_rate': 0.10168124436280626, 'subsample': 0.9925678257541187, 'colsample_bytree': 0.999484455081376, 'min_child_weight': 1, 'gamma': 1.734429223599413, 'reg_lambda': 0.004832633367915587, 'reg_alpha': 0.195081414950336, 'scale_pos_weight': 1.7670278802698338

### Training models with optimal hyperparameters

In [ ]:
# define tuned adult models (parameters hardcoded from optuna results)

models_adult_tuned = {
    "Naive Bayes": Pipeline([
        ("preprocessor", preprocessor_adult_unscaled),
        ("to_dense", FunctionTransformer(to_dense)),
        ("model", GaussianNB(
            var_smoothing=2.0413057857909265e-09
        ))
    ]),
    "Logistic Regression": Pipeline([
        ("preprocessor", preprocessor_adult_scaled),
        ("model", LogisticRegression(
            max_iter=5000,
            C=0.140759458974037,
            penalty="l2",
            solver="lbfgs",
            class_weight="balanced"
        ))
    ]),
    "Random Forest": Pipeline([
        ("preprocessor", preprocessor_adult_unscaled),
        ("model", RandomForestClassifier(
            random_state=42,
            n_estimators=400,
            max_depth=23,
            min_samples_split=15,
            min_samples_leaf=3,
            max_features=None,
            class_weight="balanced_subsample"
        ))
    ]),
    "XGBoost": Pipeline([
        ("preprocessor", preprocessor_adult_unscaled),
        ("model", XGBClassifier(
            random_state=42,
            eval_metric="logloss",
            n_estimators=800,
            max_depth=5,
            learning_rate=0.10168124436280626,
            subsample=0.9925678257541187,
            colsample_bytree=0.999484455081376,
            min_child_weight=1,
            gamma=1.734429223599413,
            reg_lambda=0.004832633367915587,
            reg_alpha=0.195081414950336,
            scale_pos_weight=1.7670278802698338
        ))
    ])
}

In [30]:
# compare tuned models with CV

compare_models_cv(models_adult_tuned, X_train_adult, y_train_adult, cv)

,model,cv_accuracy,cv_precision,cv_recall,cv_f1,cv_balanced_accuracy
3,XGBoost,0.861234,0.685707,0.775484,0.727814,0.831845
2,Random Forest,0.836588,0.621223,0.812493,0.704089,0.828330
1,Logistic Regression,0.807667,0.565426,0.849717,0.678935,0.822080
0,Naive Bayes,0.814680,0.591630,0.731736,0.653926,0.786252


### Tuned models evaluation on test data

In [31]:
# fit tuned models and evaluate on test set

trained_models_adult_tuned, test_results_adult_tuned = evaluate_models_test(
    models_adult_tuned,
    X_train_adult,
    X_test_adult,
    y_train_adult,
    y_test_adult
)

test_results_adult_tuned

,model,test_accuracy,test_precision,test_recall,test_f1,test_balanced_accuracy
2,Random Forest,0.838162,0.623410,0.817793,0.707493,0.831182
3,XGBoost,0.862934,0.692783,0.767750,0.728342,0.830316
1,Logistic Regression,0.805405,0.562518,0.840890,0.674096,0.817565
0,Naive Bayes,0.815948,0.594870,0.724123,0.653164,0.784481


### Threshold tuning for tuned models

In [32]:
# threshold tuning on tuned models (includes default 0.5)

def show_thresholds(name, model):
    probs = model.predict_proba(X_test_adult)
    df = evaluate_thresholds(y_test_adult, probs)
    top = df.sort_values("f1", ascending=False).head()
    default = df[df["threshold"] == 0.5]
    print(name)
    print(pd.concat([top, default]).drop_duplicates())
    print()

show_thresholds("Logistic Regression", trained_models_adult_tuned["Logistic Regression"])
show_thresholds("Naive Bayes", trained_models_adult_tuned["Naive Bayes"])
show_thresholds("Random Forest", trained_models_adult_tuned["Random Forest"])
show_thresholds("XGBoost", trained_models_adult_tuned["XGBoost"])

Logistic Regression
    threshold  precision    recall        f1  balanced_accuracy
10       0.60   0.628521  0.763473  0.689455           0.810750
11       0.65   0.661776  0.713858  0.686831           0.799534
9        0.55   0.593455  0.806672  0.683829           0.816403
12       0.70   0.695652  0.663815  0.679361           0.786221
8        0.50   0.562518  0.840890  0.674096           0.817565

Naive Bayes
   threshold  precision    recall        f1  balanced_accuracy
7       0.45   0.583973  0.748075  0.655916           0.790200
6       0.40   0.570025  0.769461  0.654896           0.793424
9       0.55   0.612291  0.703165  0.654589           0.781538
8       0.50   0.594870  0.724123  0.653164           0.784481
5       0.35   0.556698  0.787425  0.652259           0.795072

Random Forest
    threshold  precision    recall        f1  balanced_accuracy
10       0.60   0.687770  0.748075  0.716656           0.820613
11       0.65   0.733722  0.698888  0.715882           0.80954

### Untuned vs tuned performance

In [33]:
# compare untuned vs tuned performance

untuned_summary = test_results_adult[["model","test_accuracy","test_precision","test_recall","test_f1","test_balanced_accuracy"]].copy()
untuned_summary.columns = ["model","untuned_accuracy","untuned_precision","untuned_recall","untuned_f1","untuned_balanced_accuracy"]

tuned_summary = test_results_adult_tuned[["model","test_accuracy","test_precision","test_recall","test_f1","test_balanced_accuracy"]].copy()
tuned_summary.columns = ["model","tuned_accuracy","tuned_precision","tuned_recall","tuned_f1","tuned_balanced_accuracy"]

untuned_summary.merge(tuned_summary, on="model")

,model,untuned_accuracy,untuned_precision,untuned_recall,untuned_f1,untuned_balanced_accuracy,tuned_accuracy,tuned_precision,tuned_recall,tuned_f1,tuned_balanced_accuracy
0,Naive Bayes,0.788515,0.537841,0.826775,0.651719,0.801626,0.815948,0.594870,0.724123,0.653164,0.784481
1,XGBoost,0.874092,0.779798,0.660393,0.715146,0.800860,0.862934,0.692783,0.767750,0.728342,0.830316
2,Random Forest,0.852697,0.726448,0.616766,0.667129,0.771847,0.838162,0.623410,0.817793,0.707493,0.831182
3,Logistic Regression,0.852185,0.737513,0.593670,0.657820,0.763596,0.805405,0.562518,0.840890,0.674096,0.817565


### Comparing untuned with default threshold vs tuned with default threshold vs tuned and best threshold

In [37]:
summary_rows = []

for name in test_results_adult["model"]:
    untuned_row = test_results_adult[test_results_adult["model"] == name].iloc[0]
    tuned_row = test_results_adult_tuned[test_results_adult_tuned["model"] == name].iloc[0]

    tuned_model = trained_models_adult_tuned[name]
    probs = tuned_model.predict_proba(X_test_adult)
    threshold_table = evaluate_thresholds(y_test_adult, probs)
    best_row = threshold_table.loc[threshold_table["f1"].idxmax()]

    baseline_f1 = untuned_row["test_f1"]
    model_tuned_f1 = tuned_row["test_f1"]
    tuned_threshold_f1 = best_row["f1"]

    summary_rows.append({
        "model": name,
        "baseline_f1": round(baseline_f1, 4),
        "model_tuned_f1": round(model_tuned_f1, 4),
        "tuned_threshold": round(best_row["threshold"], 2),
        "tuned_threshold_f1": round(tuned_threshold_f1, 4),
        "gain_model": round(model_tuned_f1 - baseline_f1, 4),
        "gain_threshold": round(tuned_threshold_f1 - model_tuned_f1, 4),
        "total_gain": round(tuned_threshold_f1 - baseline_f1, 4)
    })

pd.DataFrame(summary_rows).sort_values("tuned_threshold_f1", ascending=False)

,model,baseline_f1,model_tuned_f1,tuned_threshold,tuned_threshold_f1,gain_model,gain_threshold,total_gain
1,XGBoost,0.7151,0.7283,0.55,0.7307,0.0132,0.0023,0.0155
2,Random Forest,0.6671,0.7075,0.60,0.7167,0.0404,0.0092,0.0495
3,Logistic Regression,0.6578,0.6741,0.60,0.6895,0.0163,0.0154,0.0316
0,Naive Bayes,0.6517,0.6532,0.45,0.6559,0.0014,0.0028,0.0042


### Maximizing XGBoost performance

leveraging native categorical support + imbalance weighting + tuning + threshold tuning

In [46]:
# XGBoost (native categorical handling) — hardcoded best parameters to avoid running optuna again

best_params_xgb_native = {
    "n_estimators": 800,
    "max_depth": 5,
    "learning_rate": 0.10168124436280626,
    "subsample": 0.9925678257541187,
    "colsample_bytree": 0.999484455081376,
    "min_child_weight": 1,
    "gamma": 1.734429223599413,
    "reg_lambda": 0.004832633367915587,
    "reg_alpha": 0.195081414950336,
    "scale_pos_weight": 1.7670278802698338
}

best_xgb_native = XGBClassifier(
    random_state=42,
    eval_metric="logloss",
    enable_categorical=True,   # 🔑 critical for native categorical handling
    tree_method="hist",        # 🔑 required for categorical support
    **best_params_xgb_native
)

best_xgb_native.fit(X_train_xgb, y_train_xgb)

# threshold tuning (same workflow as other models)
probs_xgb_native = best_xgb_native.predict_proba(X_test_xgb)
threshold_results_xgb_native = evaluate_thresholds(y_test_xgb, probs_xgb_native)
best_threshold_row_xgb = threshold_results_xgb_native.loc[
    threshold_results_xgb_native["f1"].idxmax()
]

preds_xgb_native = (
    probs_xgb_native[:, 1] >= best_threshold_row_xgb["threshold"]
).astype(int)

print("Best threshold:", round(best_threshold_row_xgb["threshold"], 4))
print("Final test precision:", round(precision_score(y_test_xgb, preds_xgb_native, zero_division=0), 4))
print("Final test recall:", round(recall_score(y_test_xgb, preds_xgb_native, zero_division=0), 4))
print("Final test F1:", round(f1_score(y_test_xgb, preds_xgb_native, zero_division=0), 4))
print("Final test balanced accuracy:", round(balanced_accuracy_score(y_test_xgb, preds_xgb_native), 4))

Best threshold: 0.55
Final test precision: 0.7349
Final test recall: 0.7305
Final test F1: 0.7327
Final test balanced accuracy: 0.8238


### Basic XGBoost vs Fully Leveraged XGBoost

In [47]:
# compare baseline XGB, tuned standardized XGB, and tuned native XGB

# make sure the test sets represent the same rows
assert X_test_adult.index.equals(X_test_xgb.index)
assert y_test_adult.index.equals(y_test_xgb.index)

# 1. baseline standardized XGB
probs_xgb_baseline = trained_models_adult["XGBoost"].predict_proba(X_test_adult)
preds_xgb_baseline = (probs_xgb_baseline[:, 1] >= 0.5).astype(int)
baseline_f1 = f1_score(y_test_adult, preds_xgb_baseline, zero_division=0)
baseline_bal_acc = balanced_accuracy_score(y_test_adult, preds_xgb_baseline)

# 2. tuned standardized XGB
probs_xgb_tuned = trained_models_adult_tuned["XGBoost"].predict_proba(X_test_adult)
threshold_table_xgb_tuned = evaluate_thresholds(y_test_adult, probs_xgb_tuned)
best_row_xgb_tuned = threshold_table_xgb_tuned.loc[threshold_table_xgb_tuned["f1"].idxmax()]
preds_xgb_tuned = (probs_xgb_tuned[:, 1] >= best_row_xgb_tuned["threshold"]).astype(int)
tuned_f1 = f1_score(y_test_adult, preds_xgb_tuned, zero_division=0)
tuned_bal_acc = balanced_accuracy_score(y_test_adult, preds_xgb_tuned)

# 3. tuned native XGB
probs_xgb_native = best_xgb_native.predict_proba(X_test_xgb)
threshold_table_xgb_native = evaluate_thresholds(y_test_xgb, probs_xgb_native)
best_row_xgb_native = threshold_table_xgb_native.loc[threshold_table_xgb_native["f1"].idxmax()]
preds_xgb_native = (probs_xgb_native[:, 1] >= best_row_xgb_native["threshold"]).astype(int)
native_f1 = f1_score(y_test_xgb, preds_xgb_native, zero_division=0)
native_bal_acc = balanced_accuracy_score(y_test_xgb, preds_xgb_native)

pd.DataFrame([
    {
        "model_version": "XGBoost baseline",
        "preprocessing": "standardized one-hot",
        "threshold": 0.50,
        "f1": round(baseline_f1, 4),
        "balanced_accuracy": round(baseline_bal_acc, 4),
        "f1_gain_vs_baseline": 0.0
    },
    {
        "model_version": "XGBoost tuned",
        "preprocessing": "standardized one-hot",
        "threshold": round(float(best_row_xgb_tuned["threshold"]), 4),
        "f1": round(tuned_f1, 4),
        "balanced_accuracy": round(tuned_bal_acc, 4),
        "f1_gain_vs_baseline": round(tuned_f1 - baseline_f1, 4)
    },
    {
        "model_version": "XGBoost native tuned",
        "preprocessing": "native categorical",
        "threshold": round(float(best_row_xgb_native["threshold"]), 4),
        "f1": round(native_f1, 4),
        "balanced_accuracy": round(native_bal_acc, 4),
        "f1_gain_vs_baseline": round(native_f1 - baseline_f1, 4)
    }
])

,model_version,preprocessing,threshold,f1,balanced_accuracy,f1_gain_vs_baseline
0,XGBoost baseline,standardized one-hot,0.50,0.7151,0.8009,0.0000
1,XGBoost tuned,standardized one-hot,0.55,0.7307,0.8226,0.0155
2,XGBoost native tuned,native categorical,0.55,0.7327,0.8238,0.0176


### Maximizing CatBoost performance

Leveraging native categorical support + automatic class weighting + tuning + threshold tuning and CatBoost’s built-in probability threshold setter

In [48]:
# hardcoded best CatBoost parameters so don't have to run optuna again
# optuna took about 45 minutes

best_params_cat_native = {
    "iterations": 1000,
    "depth": 6,
    "learning_rate": 0.027655486613862676,
    "l2_leaf_reg": 5.368686584281796,
    "bagging_temperature": 2.295143517707268,
    "random_strength": 3.16398880492422,
    "auto_class_weights": "SqrtBalanced"
}

best_cat_native = CatBoostClassifier(
    random_state=42,
    verbose=0,
    loss_function="Logloss",
    eval_metric="F1",
    **best_params_cat_native
)

best_cat_native.fit(X_train_cat, y_train_cat, cat_features=cat_features)

probs_cat_native = best_cat_native.predict_proba(X_test_cat)
threshold_results_cat_native = evaluate_thresholds(y_test_cat, probs_cat_native)
best_threshold_row_cat = threshold_results_cat_native.loc[
    threshold_results_cat_native["f1"].idxmax()
]

best_cat_native.set_probability_threshold(float(best_threshold_row_cat["threshold"]))
preds_cat_best = best_cat_native.predict(X_test_cat)

print("Best threshold:", round(best_threshold_row_cat["threshold"], 4))
print("Final test precision:", round(precision_score(y_test_cat, preds_cat_best, zero_division=0), 4))
print("Final test recall:", round(recall_score(y_test_cat, preds_cat_best, zero_division=0), 4))
print("Final test F1:", round(f1_score(y_test_cat, preds_cat_best, zero_division=0), 4))
print("Final test balanced accuracy:", round(balanced_accuracy_score(y_test_cat, preds_cat_best), 4))

Best threshold: 0.55
Final test precision: 0.7375
Final test recall: 0.7258
Final test F1: 0.7316
Final test balanced accuracy: 0.8223


### Fully leveraged: XGBoost vs CatBoost

In [49]:
# compare fully leveraged xgboost vs fully leveraged catboost

probs_xgb_best = best_xgb_native.predict_proba(X_test_xgb)
preds_xgb_best = (probs_xgb_best[:,1] >= best_threshold_row_xgb["threshold"]).astype(int)

probs_cat_best = best_cat_native.predict_proba(X_test_cat)
preds_cat_best = (probs_cat_best[:,1] >= best_threshold_row_cat["threshold"]).astype(int)

pd.DataFrame([
    {
        "model": "XGBoost maximized",
        "threshold": round(float(best_threshold_row_xgb["threshold"]), 4),
        "precision": round(precision_score(y_test_xgb, preds_xgb_best, zero_division=0), 4),
        "recall": round(recall_score(y_test_xgb, preds_xgb_best, zero_division=0), 4),
        "f1": round(f1_score(y_test_xgb, preds_xgb_best, zero_division=0), 4),
        "balanced_accuracy": round(balanced_accuracy_score(y_test_xgb, preds_xgb_best), 4)
    },
    {
        "model": "CatBoost maximized",
        "threshold": round(float(best_threshold_row_cat["threshold"]), 4),
        "precision": round(precision_score(y_test_cat, preds_cat_best, zero_division=0), 4),
        "recall": round(recall_score(y_test_cat, preds_cat_best, zero_division=0), 4),
        "f1": round(f1_score(y_test_cat, preds_cat_best, zero_division=0), 4),
        "balanced_accuracy": round(balanced_accuracy_score(y_test_cat, preds_cat_best), 4)
    }
])

,model,threshold,precision,recall,f1,balanced_accuracy
0,XGBoost maximized,0.55,0.7349,0.7305,0.7327,0.8238
1,CatBoost maximized,0.55,0.7375,0.7258,0.7316,0.8223


### Threshold changes for multiclass classifier

In [ ]:
# --- Generate probabilities ---
probs = model.predict_proba(X_test)

# --- Baseline prediction (default behavior) ---
preds_baseline = probs.argmax(axis=1)

# --- Choose class to focus on ---
target_class = 2   # <-- change this to your class of interest

# --- Set threshold ---
threshold = 0.3    # <-- experiment with this

# --- Apply threshold ---
preds_threshold = preds_baseline.copy()

mask = probs[:, target_class] >= threshold
preds_threshold[mask] = target_class

# --- Compare results ---
print("Baseline metric:", your_metric(y_test, preds_baseline))
print("Threshold metric:", your_metric(y_test, preds_threshold))

# --- Optional: classification report ---
from sklearn.metrics import classification_report

print("\nBaseline report:\n", classification_report(y_test, preds_baseline))
print("\nThreshold report:\n", classification_report(y_test, preds_threshold))